<a href="https://colab.research.google.com/github/Shaifali07/LLM-projects/blob/main/QuestionBankMaker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installations:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Imports

In [ ]:
!pip install pypdf
!pip install langchain_community
!pip install langchain
!pip install langchain-text-splitters
!pip install langchain_huggingface
!pip install langchain_chroma
!pip install langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
from pypdf import PdfReader
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import re
import glob
from pydantic import BaseModel, Field
from typing import List

LLM setup

In [ ]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

Extacting Units and topics from syllabus

In [ ]:
syllabus='''
[1] INTRODUCTION
Uses of computer Networks, Network Hardware-LAN,MAN,WAN, internetworks. Network Software
Design Issues, interfaces & Services, Connection Oriented & Connectionless services. Service
primitives. Relationship of services to protocols. Reference Models - OSI & TCP/IP, their comparison
& critiques.
[2] THE PHYSICAL LAYER
Transmission Media – magnetic media, twisted pair, baseband & broadband, fiber optics. Wireless
Transmission – radio, microwave, infrared & lightwave. Narrowband ISDN, Broadband ISDN &
ATM. Cellular Radio- Paging systems, cordless telephones, analog & digital telephones.
[3] THE DATA LINK LAYER
DLL Design issues, Error Detection & Correction. Elementary Data link Protocols - Utopia, Stop N
Wait, Automatic Repeat Request. Sliding Window Protocols - 1 bit sliding window, Go Back N,
Selective Repeat Protocols.
[4] MEDIUM ACCESS SUBLAYER
Channel Allocation Problem - Static & Dynamic. Multiple Access protocols - ALOHA, CSMA,
Collision Free Protocols, Limited contention protocols, WDMA protocol, wireless LAN protocols.
IEEE standards 802 for LAN & MAN - 802.2, 802.3, 802.4, 802.6 & related numericals. Bridges -
From 802.x to 802.y, transparent Bridges, Spanning Tree, Source Routing Bridges, remote bridge &
problems. Comparison of 802 bridges, High Speed LANs - FDDI, fast ethernet.
[5] THE NETWORK LAYER
Network layer Design issues. Routing Algorithms. Congestion Control Algorithms - general policies,
congestion prevention policies, traffic shaping, flow specifications, congestion control in VC subnets, choke packets, load shedding, jitter control and congestion control for malfunctioning. The network
layer in the internet - the IP protocol, IP addresses & subnets
[6] THE TRANSPORT LAYER
The Transport Service, Elements of Transport Protocols, The Internet Transport Protocols - TCP
service model, TCP protocol, TCP Segment Header, TCP Connection Management, TCP
Transmission Policy, TCP Congestion Policy. UDP & overview of Socket. Performance Issues -
Performance problems in Computer Networks (case study), Measuring Network Performance (case
study).
[7] THE APPLICATION LAYER
Network Security - Traditional Cryptography, Two Fundamental Cryptographic Principles, Secret-Key
Algorithms, Public- key Algorithms, Authentication protocols, Digital Signatures, Social Issues.,
E-mail (case study), SNMP (case study).'''

In [ ]:
llm2=ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0)

In [ ]:
class Topic_Formation(BaseModel):
  Units: str = Field(description="Unit extracted from the syllabus")
  topics: str = Field(description="Key topics form the unit")
parser_topic=JsonOutputParser(pydantic_object=Topic_Formation)
format_instructions_topic = parser_topic.get_format_instructions()

In [ ]:
prompt_topic = ChatPromptTemplate.from_messages([
    ("system","You are an academic assistant. Your task is to extract structured units from the syllabus."),
    ("human",'''
Instructions:
1. Identify all units/modules from the syllabus {syllabus}.
2. For each unit, extract:
   - unit_name
   - key topics (list of concepts inside the unit)
3. Clean and organize properly.

Output format instructions  {format_instructions_topic}

'''

)])
chain = prompt_topic | llm2 |parser_topic
topics = chain.invoke({"syllabus":syllabus,"format_instructions_topic":format_instructions_topic})
print(topics)

{'Units': [{'unit_name': 'INTRODUCTION', 'topics': 'Uses of computer Networks, Network Hardware-LAN,MAN,WAN, internetworks. Network Software Design Issues, interfaces & Services, Connection Oriented & Connectionless services. Service primitives. Relationship of services to protocols. Reference Models - OSI & TCP/IP, their comparison & critiques.'}, {'unit_name': 'THE PHYSICAL LAYER', 'topics': 'Transmission Media – magnetic media, twisted pair, baseband & broadband, fiber optics. Wireless Transmission – radio, microwave, infrared & lightwave. Narrowband ISDN, Broadband ISDN & ATM. Cellular Radio- Paging systems, cordless telephones, analog & digital telephones.'}, {'unit_name': 'THE DATA LINK LAYER', 'topics': 'DLL Design issues, Error Detection & Correction. Elementary Data link Protocols - Utopia, Stop N Wait, Automatic Repeat Request. Sliding Window Protocols - 1 bit sliding window, Go Back N, Selective Repeat Protocols.'}, {'unit_name': 'MEDIUM ACCESS SUBLAYER', 'topics': 'Channel 

In [ ]:
unit_names=[]
for t in topics["Units"]:
  unit_names.append(t["unit_name"])
print(unit_names)

['INTRODUCTION', 'THE PHYSICAL LAYER', 'THE DATA LINK LAYER', 'MEDIUM ACCESS SUBLAYER', 'THE NETWORK LAYER', 'THE TRANSPORT LAYER', 'THE APPLICATION LAYER']


Extracting Questions from previous papers

In [52]:
llm=ChatGroq(
    model_name="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0)

In [ ]:
class Clean_text(BaseModel):
  question: str = Field(description="Question extracted from papers")
  marks: int | None = Field(description="Marks for each question ")
  unit: str | None = Field(description="Unit of each question from syllbus ")
parser_clean_text=JsonOutputParser(pydantic_object=Clean_text)
format_instructions_clean_text = parser_clean_text.get_format_instructions()

In [53]:
prompt_cleaner= ChatPromptTemplate.from_messages ([
    ("system", "You are an expert at processing university exam papers."),
    ("human",'''
    You are given
1. raw extracted text from an exam paper. The text contains noise such as:
headers (university name, subject, date, etc.)
instructions
question numbers (Q.1, Q.2, etc.)
course outcome tags like CO1, CO2
marks like [2], [6]
formatting issues
"OR" sections
mixed sub-questions like (a), (b), (i), (ii)

Your task is to extract clean, meaningful questions.
2. Syllabus units

----------------------
SYLLABUS:
{syllabus}
----------------------
Your tasks:

### Task 1: Clean Questions
Instructions:
Remove all irrelevant text such as:
headers
instructions
page numbers
"Attempt any", "Do as directed", etc.
Extract ONLY actual questions.
keep all sub-questions into the same parent  question:
(a), (b), (c)
(i), (ii), (iii)
numbered lists like 1., 2.
If a question contains multiple logical parts, split them into separate questions.
Remove "OR" and treat alternative questions as independent questions.
Preserve the full meaning of each question.
Extract marks if available, otherwise set marks = null.
Clean formatting:
remove extra spaces
fix broken sentences
### Task 2: Tag Each Question
For each question, assign:
- unit (from syllabus)
Instructions:
- Choose the BEST matching unit
- Do NOT invent new units
- Use only provided syllabus
-topics for each units are {unit_topics} you can use them to decide the unit of the question. You will just return unit and marks for each question not topics.
Output format (STRICT JSON): as per {format_instructions_clean_text}

[
{{
"question": "clean question text",
"marks": number or null
"unit": "unit name",
}}
]

Important:
Do NOT include any explanation.
Do NOT include headers or instructions.
Return ONLY valid JSON.
Ensure each question is complete and readable.

Now process the following text:

{input_text}''')
])
chain=prompt_cleaner | llm|parser_clean_text

In [54]:
folder_path = '/content/drive/My Drive/pdf_papers/'
pdf_files = glob.glob(os.path.join(folder_path, '*.pdf'))
text = []

for file_path in pdf_files:

    with open(file_path, 'rb') as pdf_file_obj:
        reader = PdfReader(pdf_file_obj)
        file_text=' '

        for page in reader.pages:
            file_text += page.extract_text() or ""

        print(f"Read file: {os.path.basename(file_path)}")
        text.append(file_text)



Read file: cn 1st int 2024.pdf
Read file: cn_third_sessional_2024.pdf
Read file: cn_updated_second_sessional_2024.pdf
Read file: cn_ext_paper1.pdf
Read file: cn_rem_paper1..pdf
Read file: CN_1-Sessional_30-12-2025.docx.pdf
Read file: cn_sess2_2026.pdf
Read file: cn_sess3_2026.pdf


In [55]:
result=[]
import time

for i,t in enumerate(text):
    file_text=t
    print(f"Processing file {i+1}...")
    time.sleep(0.2)
    result+=chain.invoke({"input_text":file_text,"syllabus":unit_names,"unit_topics": topics["Units"],"format_instructions_clean_text":format_instructions_clean_text})


Processing file 1...
Processing file 2...
Processing file 3...
Processing file 4...
Processing file 5...
Processing file 6...
Processing file 7...
Processing file 8...


In [56]:
print(result)

[{'question': 'What is sender’s window in case of selective repeat, go back n, and stop and wait protocol if the sequence number is 4 bits long?', 'marks': 2, 'unit': 'THE DATA LINK LAYER'}, {'question': 'How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00 and 0x95?', 'marks': 2, 'unit': 'THE DATA LINK LAYER'}, {'question': 'The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1. Does the received data have an error? (show the procedure)', 'marks': 2, 'unit': 'THE DATA LINK LAYER'}, {'question': 'A supernet is created by combining some class C blocks. It has a first address of 205.16.32.0 and a supernet mask of 255.255.248.0. How many Class C blocks are there in this supernet? Explain.', 'marks': 2, 'unit': 'THE NETWORK LAYER'}, {'question': 'An IPv4 packet has arrived with the first 8 bits as shown:  01000111... How many bytes of options are being carried by this packet? Show your calculation.', 'marks': 2, 'unit': 'THE NETW

In [58]:
from collections import defaultdict

grouped = defaultdict(list)

for q in result:   # your list of dicts
    unit_name = q.get("unit", "Uncategorized") # Use .get() to handle missing 'unit' key
    grouped[unit_name].append(q)

In [60]:
for unit, questions in grouped.items():
    print(f"\n===== {unit} =====")

    for i, q in enumerate(questions, 1):
        marks_display = q.get('marks', '--') # Safely get marks, default to 'N/A' if not found
        print(f"{i}. {q['question']} ({marks_display} marks)")


===== THE DATA LINK LAYER =====
1. What is sender’s window in case of selective repeat, go back n, and stop and wait protocol if the sequence number is 4 bits long? (2 marks)
2. How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00 and 0x95? (2 marks)
3. The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1. Does the received data have an error? (show the procedure) (2 marks)
4. What is error control? Where is it provided and where is it mandatory? What provisions are necessary for error detection? (6 marks)
5. The source has to send a message consisting of 13 frames to destination using go back n(sender’s window size=3). All frames are ready and immediately available for transmission. What is the number of frames that source will transmit for sending the message to destination in the following cases? 1. Every 5th frame that source transmits gets lost (but no acknowledgement is lost) 2. Every 4th frame that source transmits g